In [1]:
# =========================
# 📦 1. IMPORTS
# =========================
import pandas as pd
import xmlrpc.client
import re
import sqlite3

In [2]:
# 1. Conectar ao arquivo .db
conexao = sqlite3.connect( r"C:\Users\tiago.premiano\Downloads\banco_precos.db")
produtos = pd.read_sql_query("SELECT * FROM produtos", conexao)

# 4. Fechar a conexão
conexao.close()

In [8]:
import xmlrpc.client
import pandas as pd

# ── CONEXÃO ──────────────────────────────────────────────────────────────────
url      = "https://crm.brainess.com.br/"
db       = "sohome_18"
username = "tiago@sohome.com"
password = "admin"

common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common")
uid    = common.authenticate(db, username, password, {})
models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object")

print(f"✅ Conectado como uid={uid}")


def cadastrar_produtos_odoo(df):
    # Pega uma linha por produto (sem duplicatas)
    produtos_df = df

    print(f"   Produtos a cadastrar: {len(produtos_df)}")

    # ── HELPER: busca ou cria um registro ────────────────────────────────────
    def get_or_create(model, domain, values):
        ids = models.execute_kw(db, uid, password, model, "search", [domain])
        if ids:
            return ids[0]
        return models.execute_kw(db, uid, password, model, "create", [values])

    # ── 1. CATEGORIAS (product.category) ─────────────────────────────────────
    print("\n📂 Sincronizando categorias...")
    categoria_ids = {}

    for cat_name in produtos_df["classificacao"].dropna().unique():
        cat_name = str(cat_name).strip()
        cat_id = get_or_create(
            "product.category",
            [["name", "=", cat_name]],
            {"name": cat_name},
        )
        categoria_ids[cat_name] = cat_id
        print(f"   Categoria '{cat_name}' → id={cat_id}")

    # ── 2. MARCAS (product.brand) ─────────────────────────────────────────────
    print("\n🏷️  Sincronizando marcas...")
    marca_ids = {}

    for marca_name in produtos_df["marca"].dropna().unique():
        marca_name = str(marca_name).strip()
        brand_id = get_or_create(
            "product.brand",
            [["name", "=", marca_name]],
            {"name": marca_name},
        )
        marca_ids[marca_name] = brand_id
        print(f"   Marca '{marca_name}' → id={brand_id}")

    # ── 3. PRODUTOS (product.template) ───────────────────────────────────────
    print("\n📦 Cadastrando produtos...")

    criados    = 0
    existentes = 0
    erros      = 0

    for _, row in produtos_df.iterrows():
        nome       = str(row["produto"]).strip()
        codigo     = str(row["codigo"]).strip() if pd.notna(row["codigo"]) else ""
        marca_nome = str(row["marca"]).strip() if pd.notna(row["marca"]) else ""
        cat_nome   = str(row["classificacao"]).strip() if pd.notna(row["classificacao"]) else ""

        domain = [["name", "=", nome]]
        if codigo:
            domain = ["|", ["default_code", "=", codigo], ["name", "=", nome]]

        ids_exist = models.execute_kw(db, uid, password, "product.template", "search", [domain])

        vals = {
            "name":         nome,
            "default_code": codigo,
            "type":         "consu",
            "sale_ok":      True,
            "purchase_ok":  False,
        }

        if cat_nome and cat_nome in categoria_ids:
            vals["categ_id"] = categoria_ids[cat_nome]

        if marca_nome and marca_nome in marca_ids:
            vals["brand_id"] = marca_ids[marca_nome]  # many2one, passa só o id

        try:
            if ids_exist:
                models.execute_kw(db, uid, password, "product.template", "write", [ids_exist, vals])
                existentes += 1
                print(f"   ↻ Atualizado: {nome}")
            else:
                new_id = models.execute_kw(db, uid, password, "product.template", "create", [vals])
                criados += 1
                print(f"   ✅ Criado: {nome} (id={new_id})")
        except Exception as e:
            erros += 1
            print(f"   ❌ Erro em '{nome}': {e}")

    # ── RESUMO ────────────────────────────────────────────────────────────────
    print(f"\n{'='*50}")
    print(f"✅ Criados:     {criados}")
    print(f"↻  Atualizados: {existentes}")
    print(f"❌ Erros:       {erros}")
    print(f"{'='*50}")


# ── USO ───────────────────────────────────────────────────────────────────────
# Substitua pelo seu df já carregado do banco:
# df = pd.merge(variantes, produtos, on="produto", how="left")
# cadastrar_produtos_odoo(df)

✅ Conectado como uid=2


In [9]:
cadastrar_produtos_odoo(produtos)

   Produtos a cadastrar: 327

📂 Sincronizando categorias...
   Categoria 'POLTRONAS' → id=4
   Categoria 'SOFAS LIVING' → id=5
   Categoria 'PUFFS' → id=6
   Categoria 'BANQUETA' → id=7
   Categoria 'SOFAS LIVING CURVO' → id=8
   Categoria 'APARADOR' → id=9
   Categoria 'BANCO' → id=10
   Categoria 'SOFAS ARTICULADOS' → id=11
   Categoria 'BIOMBO' → id=12
   Categoria 'BUFFET' → id=13
   Categoria 'CADEIRA' → id=14
   Categoria 'CAMAS' → id=15
   Categoria 'CAPA' → id=16
   Categoria 'ESTANTE' → id=17
   Categoria 'MESA' → id=18
   Categoria 'MESA DE JANTAR' → id=19
   Categoria 'MESA AP' → id=20
   Categoria 'MESA DE CENTRO' → id=21
   Categoria 'MOSTRUARIO' → id=22
   Categoria 'ESPELHO' → id=23
   Categoria 'TECIDO' → id=24

🏷️  Sincronizando marcas...
   Marca 'CENTURY' → id=1
   Marca 'PONTOVIRGULA' → id=2

📦 Cadastrando produtos...
   ✅ Criado: ABACO (id=4)
   ✅ Criado: ACLIVE (id=5)
   ✅ Criado: ADANA (id=6)
   ✅ Criado: AFAGO (id=7)
   ✅ Criado: AGHATA PLUS (id=8)
   ✅ Criado: 

In [5]:
fields = models.execute_kw(db, uid, password, "product.template", "fields_get", [], {"attributes": ["string", "type"]})

for fname, finfo in sorted(fields.items()):
    if any(x in fname.lower() or x in finfo["string"].lower() for x in ["tag", "marca", "brand"]):
        print(f"{fname:30} | {finfo['type']:15} | {finfo['string']}")

account_tag_ids                | many2many       | Account Tags
brand_id                       | many2one        | Brand
contract_brand_id              | many2one        | Marca
product_tag_ids                | many2many       | Tags
